# Notebook 61 — Quadratic Latent Transport vs Static Symbolic Chart

Notebook 60 showed that affine latent ODE paths were too straight.
This notebook tests one final dynamics upgrade:

\[
\frac{dz}{dk}=A z + Q(z,z) + b
\]

against the current best predictive model:

\[
\beta = f(\text{metadata}, k)
\]

## Central question

Does adding latent curvature close the gap to static symbolic transport,
or is static symbolic the final coordinate chart?

## Models compared

1. Static symbolic coefficient chart
2. Global affine latent transport
3. Global quadratic latent transport
4. Forcing-conditioned affine latent transport
5. Forcing-conditioned quadratic latent transport
6. Flow-conditioned quadratic latent transport
7. Forcing × flow-conditioned quadratic latent transport
8. Drift-only baselines

## Evaluation

Forward:

\[
3 \to 5 \to 7
\]

Backward:

\[
7 \to 5 \to 3
\]

Metrics:

- latent RMSE
- coefficient RMSE
- field RMSE
- trajectory RMSE

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed governing template and per-regime coefficients

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

## Latent coefficient space

Use `latent_dim = 3` as the main space, because Notebook 57 showed that three components capture nearly all coefficient variance.

In [ ]:
# Latent encoder/decoder

LATENT_DIM = 3

coef_scaler = StandardScaler()
Y = coef_df[COEF_COLS].to_numpy(dtype=float)
Ystd = coef_scaler.fit_transform(Y)

pca = PCA(n_components=LATENT_DIM, random_state=42)
Z = pca.fit_transform(Ystd)

print("Explained variance:", pca.explained_variance_ratio_, "sum=", pca.explained_variance_ratio_.sum())

for j in range(LATENT_DIM):
    coef_df[f"z{j+1}"] = Z[:, j]

def encode_beta(beta):
    beta = np.asarray(beta, dtype=float).reshape(1, -1)
    return pca.transform(coef_scaler.transform(beta))[0]

def decode_z(z):
    z = np.asarray(z, dtype=float).reshape(1, -1)
    return coef_scaler.inverse_transform(pca.inverse_transform(z))[0]

# 2D plot coordinates use first two PCs
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, ["forcing_mode", "flow_mode", "system"]):
    for val in sorted(coef_df[col].astype(str).unique()):
        sub = coef_df[coef_df[col].astype(str) == val]
        ax.scatter(sub["z1"], sub["z2"], label=val, s=40)
    ax.set_xlabel("z1")
    ax.set_ylabel("z2")
    ax.set_title(f"Latent manifold by {col}")
    ax.legend()

plt.tight_layout()
plt.show()

## Static symbolic baseline from Notebook 58

In [ ]:
def build_static_symbolic_features(df_in, columns=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)

    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")

    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")

    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

def fit_static_symbolic(train_coef_df, test_coef_df):
    X_train = build_static_symbolic_features(train_coef_df)
    X_test = build_static_symbolic_features(test_coef_df, columns=X_train.columns)

    Yhat = np.zeros((len(test_coef_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        y_train = train_coef_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(X_train)
        Xte_s = scaler.transform(X_test)

        model = LassoCV(cv=min(5, len(train_coef_df)), random_state=42, max_iter=20000)
        model.fit(Xtr_s, y_train)
        Yhat[:, j] = model.predict(Xte_s)

    return Yhat

## Build latent k-trajectories and finite-difference transport samples

In [ ]:
traj_group_cols = ["system", "task", "forcing_mode", "flow_mode"]

def build_latent_trajectories(coef_table):
    trajectories = []
    for vals, sub in coef_table.groupby(traj_group_cols):
        sub = sub.sort_values("k").reset_index(drop=True)
        if sub["k"].nunique() < 2:
            continue

        betas = sub[COEF_COLS].to_numpy(dtype=float)
        zs = sub[[f"z{j+1}" for j in range(LATENT_DIM)]].to_numpy(dtype=float)
        ks = sub["k"].to_numpy(dtype=float)

        trajectories.append({
            "meta": vals,
            "system": vals[0],
            "task": vals[1],
            "forcing_mode": vals[2],
            "flow_mode": vals[3],
            "condition_key": vals[2] + "__x__" + vals[3],
            "k": ks,
            "beta": betas,
            "z": zs,
            "rows": sub.copy(),
        })
    return trajectories

trajectories = build_latent_trajectories(coef_df)

def build_transport_df(trajectories):
    rows = []
    for tr in trajectories:
        ks = tr["k"]
        zs = tr["z"]
        rows_df = tr["rows"]

        for i in range(len(ks) - 1):
            k0, k1 = ks[i], ks[i + 1]
            dk = k1 - k0
            z0, z1 = zs[i], zs[i + 1]
            dzdk = (z1 - z0) / dk

            row = {
                "system": tr["system"],
                "task": tr["task"],
                "forcing_mode": tr["forcing_mode"],
                "flow_mode": tr["flow_mode"],
                "condition_key": tr["condition_key"],
                "k0": k0,
                "k1": k1,
                "dk": dk,
                "regime_id_0": rows_df.loc[i, "regime_id"],
                "regime_id_1": rows_df.loc[i + 1, "regime_id"],
            }

            for j in range(LATENT_DIM):
                row[f"z{j+1}_0"] = z0[j]
                row[f"dz{j+1}_dk"] = dzdk[j]

            rows.append(row)
    return pd.DataFrame(rows)

transport_df = build_transport_df(trajectories)
display(transport_df.head())
print("Transport samples:", len(transport_df))

## Transport feature maps

Affine:

\[
\phi(z)=[z_1,z_2,z_3]
\]

Quadratic:

\[
\phi(z)=[z_1,z_2,z_3,z_1^2,z_1z_2,z_1z_3,z_2^2,z_2z_3,z_3^2]
\]

In [ ]:
Z_COLS = [f"z{j+1}_0" for j in range(LATENT_DIM)]
DZ_COLS = [f"dz{j+1}_dk" for j in range(LATENT_DIM)]

def latent_features_from_z(z, mode="affine"):
    z = np.asarray(z, dtype=float)
    feats = list(z)

    if mode == "quadratic":
        for i in range(len(z)):
            for j in range(i, len(z)):
                feats.append(z[i] * z[j])

    if mode == "drift":
        feats = [1.0]

    return np.asarray(feats, dtype=float)

def latent_feature_matrix(df_in, mode="affine"):
    X = []
    for _, row in df_in.iterrows():
        z = row[Z_COLS].to_numpy(dtype=float)
        X.append(latent_features_from_z(z, mode=mode))
    return np.vstack(X)

def feature_names(mode="affine"):
    names = [f"z{j+1}" for j in range(LATENT_DIM)]
    if mode == "quadratic":
        for i in range(LATENT_DIM):
            for j in range(i, LATENT_DIM):
                names.append(f"z{i+1}z{j+1}")
    if mode == "drift":
        names = ["1"]
    return names

print("affine features:", feature_names("affine"))
print("quadratic features:", feature_names("quadratic"))

## Fit latent transport models

In [ ]:
def fit_conditioned_flow(train_df, condition_col=None, mode="affine", alpha=1.0, min_samples=3):
    models = {}
    fallback = {}

    X_all = latent_feature_matrix(train_df, mode=mode)
    Y_all = train_df[DZ_COLS].to_numpy(dtype=float)

    fallback_model = Ridge(alpha=alpha)
    fallback_model.fit(X_all, Y_all)
    fallback = {"model": fallback_model, "mode": mode, "n": len(train_df), "key": "__fallback__"}

    if condition_col is None:
        models["__global__"] = fallback
        return models, fallback

    for key, sub in train_df.groupby(condition_col):
        if len(sub) < min_samples:
            continue
        X = latent_feature_matrix(sub, mode=mode)
        Y = sub[DZ_COLS].to_numpy(dtype=float)
        model = Ridge(alpha=alpha)
        model.fit(X, Y)
        models[key] = {"model": model, "mode": mode, "n": len(sub), "key": key}

    return models, fallback

def predict_dz(models, fallback, z, key=None):
    if key in models:
        pack = models[key]
    else:
        pack = fallback

    X = latent_features_from_z(z, mode=pack["mode"]).reshape(1, -1)
    return pack["model"].predict(X)[0]

def integrate_latent(z_start, k_start, k_targets, models, fallback, key=None):
    z = np.asarray(z_start, dtype=float).copy()
    k_curr = float(k_start)
    path = {k_curr: z.copy()}

    for k_next in k_targets:
        dk = float(k_next - k_curr)
        dz = predict_dz(models, fallback, z, key=key)
        z = z + dk * dz
        k_curr = float(k_next)
        path[k_curr] = z.copy()

    return path

## Multi-step evaluation

In [ ]:
def regime_at_k(tr, kval):
    idxs = np.where(np.isclose(tr["k"], kval))[0]
    if len(idxs) == 0:
        return None
    i = int(idxs[0])
    return tr["rows"].loc[i, "regime_id"], tr["beta"][i], tr["z"][i]

def eval_beta_prediction(rid, beta_true, beta_hat):
    sub = regime_subsets[rid]
    y_emp = sub["predicted_flow"].to_numpy(dtype=float)
    pred = predict_with_beta(sub, beta_hat)
    return {
        "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
        "field_rmse": math.sqrt(mean_squared_error(y_emp, pred)),
        "traj_rmse": trajectory_gap(sub, beta_true, beta_hat),
    }

model_specs = [
    ("global_affine", None, "affine", "__global__"),
    ("global_quadratic", None, "quadratic", "__global__"),
    ("forcing_affine", "forcing_mode", "affine", "forcing_mode"),
    ("forcing_quadratic", "forcing_mode", "quadratic", "forcing_mode"),
    ("flow_quadratic", "flow_mode", "quadratic", "flow_mode"),
    ("forcing_flow_quadratic", "condition_key", "quadratic", "condition_key"),
    ("forcing_drift", "forcing_mode", "drift", "forcing_mode"),
]

rows = []
param_rows = []

for direction in ["forward_3_to_7", "backward_7_to_3"]:
    for tr in trajectories:
        ks = list(tr["k"])
        if not all(k in ks for k in [3.0, 5.0, 7.0]):
            continue

        if direction == "forward_3_to_7":
            k_start = 3.0
            k_targets = [5.0, 7.0]
        else:
            k_start = 7.0
            k_targets = [5.0, 3.0]

        start = regime_at_k(tr, k_start)
        if start is None:
            continue

        start_rid, beta_start, z_start = start
        excluded_rids = set(tr["rows"]["regime_id"].tolist())

        train_transport = transport_df[
            ~transport_df["regime_id_0"].isin(excluded_rids)
            & ~transport_df["regime_id_1"].isin(excluded_rids)
        ].reset_index(drop=True)

        if len(train_transport) < 8:
            train_transport = transport_df.copy()

        train_coef = coef_df[~coef_df["regime_id"].isin(excluded_rids)].reset_index(drop=True)
        if len(train_coef) < 8:
            train_coef = coef_df.copy()

        for model_name, condition_col, mode, key_source in model_specs:
            models, fallback = fit_conditioned_flow(
                train_transport,
                condition_col=condition_col,
                mode=mode,
                alpha=1.0,
                min_samples=3,
            )

            if key_source == "__global__":
                key = "__global__"
            elif key_source == "forcing_mode":
                key = tr["forcing_mode"]
            elif key_source == "flow_mode":
                key = tr["flow_mode"]
            elif key_source == "condition_key":
                key = tr["condition_key"]
            else:
                key = None

            z_path = integrate_latent(z_start, k_start, k_targets, models, fallback, key=key)

            # model complexity / overfit guard bookkeeping
            n_features = len(feature_names(mode))
            n_targets = LATENT_DIM
            n_params = n_features * n_targets + n_targets
            family_n = models[key]["n"] if key in models else fallback["n"]
            param_rows.append({
                "model": model_name,
                "direction": direction,
                "source_regime": start_rid,
                "family_key": key,
                "mode": mode,
                "family_n": family_n,
                "n_features": n_features,
                "n_params": n_params,
                "params_per_sample": n_params / max(family_n, 1),
            })

            for kt in k_targets:
                target = regime_at_k(tr, kt)
                if target is None:
                    continue
                rid, beta_true, z_true = target
                beta_hat = decode_z(z_path[float(kt)])
                metrics = eval_beta_prediction(rid, beta_true, beta_hat)
                metrics.update({
                    "direction": direction,
                    "model": model_name,
                    "source_regime": start_rid,
                    "target_regime": rid,
                    "target_k": kt,
                    "latent_rmse": math.sqrt(mean_squared_error(z_true, z_path[float(kt)])),
                })
                rows.append(metrics)

        # static symbolic baseline
        target_rows = []
        target_info = []
        for kt in k_targets:
            target = regime_at_k(tr, kt)
            if target is not None:
                rid, beta_true, z_true = target
                target_rows.append(coef_df[coef_df["regime_id"] == rid])
                target_info.append((kt, rid, beta_true, z_true))

        if len(target_rows) > 0:
            test_coef = pd.concat(target_rows, axis=0).reset_index(drop=True)
            beta_static_mat = fit_static_symbolic(train_coef, test_coef)

            for idx, (kt, rid, beta_true, z_true) in enumerate(target_info):
                beta_hat = beta_static_mat[idx]
                z_hat = encode_beta(beta_hat)
                metrics = eval_beta_prediction(rid, beta_true, beta_hat)
                metrics.update({
                    "direction": direction,
                    "model": "static_symbolic",
                    "source_regime": start_rid,
                    "target_regime": rid,
                    "target_k": kt,
                    "latent_rmse": math.sqrt(mean_squared_error(z_true, z_hat)),
                })
                rows.append(metrics)

eval_df = pd.DataFrame(rows)
param_df = pd.DataFrame(param_rows)

display(eval_df.head())
display(param_df.head())

In [ ]:
summary_df = eval_df.groupby(["direction", "model"])[
    ["coef_rmse", "field_rmse", "traj_rmse", "latent_rmse"]
].mean().reset_index()

display(summary_df.sort_values(["direction", "traj_rmse"]))

In [ ]:
# Model comparison plots

for metric in ["coef_rmse", "field_rmse", "traj_rmse", "latent_rmse"]:
    pivot = summary_df.pivot_table(index="direction", columns="model", values=metric)
    ax = pivot.plot(kind="bar", figsize=(15, 5))
    ax.set_title(metric)
    ax.set_ylabel(metric)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## Overfit guard

Quadratic conditioned flows can have more parameters than samples inside each family. 
This table highlights risky model-family fits.

In [ ]:
overfit_summary = param_df.groupby(["model", "mode"])[
    ["family_n", "n_features", "n_params", "params_per_sample"]
].mean().reset_index().sort_values("params_per_sample", ascending=False)

display(overfit_summary)

plt.figure(figsize=(10, 4))
plt.bar(overfit_summary["model"], overfit_summary["params_per_sample"])
plt.axhline(1.0, linestyle="--")
plt.ylabel("parameters per family sample")
plt.title("Overfit guard: average parameters per sample")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Curvature diagnostic

Compare true latent path, affine prediction, quadratic prediction, and static symbolic decoded points.

In [ ]:
# Pick representative trajectory where quadratic improves most over affine among latent models

case = eval_df.pivot_table(
    index=["direction", "source_regime", "target_regime", "target_k"],
    columns="model",
    values="traj_rmse",
    aggfunc="mean",
).reset_index()

if "forcing_affine" in case.columns and "forcing_quadratic" in case.columns:
    case["quad_gain"] = case["forcing_affine"] - case["forcing_quadratic"]
    rep_case = case.sort_values("quad_gain", ascending=False).iloc[0]
else:
    rep_case = case.iloc[0]

source_regime = rep_case["source_regime"]
direction = rep_case["direction"]

# Find the original trajectory
rep_tr = None
for tr in trajectories:
    if source_regime in set(tr["rows"]["regime_id"].tolist()):
        rep_tr = tr
        break

if direction == "forward_3_to_7":
    k_start = 3.0
    k_targets = [5.0, 7.0]
else:
    k_start = 7.0
    k_targets = [5.0, 3.0]

start_rid, beta_start, z_start = regime_at_k(rep_tr, k_start)
excluded_rids = set(rep_tr["rows"]["regime_id"].tolist())
train_transport = transport_df[
    ~transport_df["regime_id_0"].isin(excluded_rids)
    & ~transport_df["regime_id_1"].isin(excluded_rids)
].reset_index(drop=True)
if len(train_transport) < 8:
    train_transport = transport_df.copy()

# Forcing affine and quadratic
models_aff, fallback_aff = fit_conditioned_flow(train_transport, "forcing_mode", "affine", alpha=1.0)
models_quad, fallback_quad = fit_conditioned_flow(train_transport, "forcing_mode", "quadratic", alpha=1.0)

key = rep_tr["forcing_mode"]

path_aff = integrate_latent(z_start, k_start, k_targets, models_aff, fallback_aff, key=key)
path_quad = integrate_latent(z_start, k_start, k_targets, models_quad, fallback_quad, key=key)

# Static symbolic decoded points
train_coef = coef_df[~coef_df["regime_id"].isin(excluded_rids)].reset_index(drop=True)
if len(train_coef) < 8:
    train_coef = coef_df.copy()

target_rows = []
target_info = []
for kt in k_targets:
    target = regime_at_k(rep_tr, kt)
    if target is not None:
        rid, beta_true, z_true = target
        target_rows.append(coef_df[coef_df["regime_id"] == rid])
        target_info.append((kt, rid, beta_true, z_true))

test_coef = pd.concat(target_rows, axis=0).reset_index(drop=True)
beta_static_mat = fit_static_symbolic(train_coef, test_coef)
static_z = {float(k_start): z_start}
for idx, (kt, rid, beta_true, z_true) in enumerate(target_info):
    static_z[float(kt)] = encode_beta(beta_static_mat[idx])

true_z = {float(k): z for k, z in zip(rep_tr["k"], rep_tr["z"])}

plt.figure(figsize=(8, 6))

true_points = np.array([true_z[k] for k in sorted(true_z.keys())])
plt.plot(true_points[:, 0], true_points[:, 1], marker="o", label="true path")

aff_points = np.array([path_aff[k] for k in sorted(path_aff.keys())])
plt.plot(aff_points[:, 0], aff_points[:, 1], marker="x", linestyle="--", label="forcing affine")

quad_points = np.array([path_quad[k] for k in sorted(path_quad.keys())])
plt.plot(quad_points[:, 0], quad_points[:, 1], marker="s", linestyle="--", label="forcing quadratic")

stat_points = np.array([static_z[k] for k in sorted(static_z.keys())])
plt.plot(stat_points[:, 0], stat_points[:, 1], marker="^", linestyle=":", label="static symbolic decoded")

for k, z in true_z.items():
    plt.text(z[0], z[1], f"k={int(k)}", fontsize=9)

plt.title(f"Curvature diagnostic: {rep_tr['forcing_mode']} | {rep_tr['flow_mode']} | {direction}")
plt.xlabel("z1")
plt.ylabel("z2")
plt.legend()
plt.tight_layout()
plt.show()

## Learned quadratic operator inspection

For one forcing family, show the quadratic feature coefficients for each latent derivative.

In [ ]:
# Fit forcing quadratic on all data and inspect coefficients

models_quad_all, fallback_quad_all = fit_conditioned_flow(
    transport_df,
    condition_col="forcing_mode",
    mode="quadratic",
    alpha=1.0,
    min_samples=3,
)

feat_names = feature_names("quadratic")

for key, pack in models_quad_all.items():
    model = pack["model"]
    coef_mat = pd.DataFrame(
        model.coef_,
        index=[f"dz{j+1}/dk" for j in range(LATENT_DIM)],
        columns=feat_names,
    )
    print("forcing:", key, "n=", pack["n"])
    display(coef_mat.round(4))

## Final decision table

Quadratic latent transport is worth continuing only if it gets close to static symbolic:

\[
\text{traj RMSE}_{quad} \leq 1.5 \times \text{traj RMSE}_{static}
\]

or beats static on at least one direction.

In [ ]:
decision_rows = []

for direction, sub in summary_df.groupby("direction"):
    static = sub[sub["model"] == "static_symbolic"].iloc[0]
    best_quad = sub[sub["model"].str.contains("quadratic")].sort_values("traj_rmse").iloc[0]
    best_latent = sub[sub["model"] != "static_symbolic"].sort_values("traj_rmse").iloc[0]
    best_overall = sub.sort_values("traj_rmse").iloc[0]

    ratio = best_quad["traj_rmse"] / max(static["traj_rmse"], 1e-12)
    if best_quad["traj_rmse"] <= static["traj_rmse"]:
        verdict = "quadratic latent beats static"
    elif ratio <= 1.5:
        verdict = "quadratic latent competitive"
    else:
        verdict = "static symbolic remains final chart"

    decision_rows.append({
        "direction": direction,
        "best_overall": best_overall["model"],
        "best_latent": best_latent["model"],
        "best_quadratic": best_quad["model"],
        "static_traj_rmse": static["traj_rmse"],
        "best_quad_traj_rmse": best_quad["traj_rmse"],
        "quad_to_static_ratio": ratio,
        "verdict": verdict,
    })

decision_df = pd.DataFrame(decision_rows)
display(decision_df)

## Working conclusion

Notebook 61 tests whether adding curvature to latent transport closes the gap with the static symbolic chart.

Interpretation guide:

- If quadratic latent transport beats or nearly matches static symbolic, continue to operator analysis.
- If static symbolic remains much better, stop expanding latent dynamics and formalize the static symbolic law as the final coordinate chart.

Recommended next notebook:

- If quadratic wins: `62_transport_operator_analysis_and_symbolic_reduction.ipynb`
- If static wins: `62_static_symbolic_law_as_final_coordinate_chart.ipynb`